# ECS vs Jaccard Statistical Evaluation

**Artifact:** ECS vs Jaccard Statistical Evaluation  
**Dataset:** 605 synthetic near-duplicate text pairs (300 near_dup, 5 hard_neg, 300 random)  
**Method:** Edit Clustering Score (ECS) — a spatial-clustering measure of how clustered character-level edit positions are in a text pair. Compared against Jaccard overlap for near-duplicate detection.

**Key question:** Does inverted-ECS (lower ECS = more near-duplicate) complement Jaccard for detecting near-duplicates?

**Results summary:**
- Inverted-ECS AUC on hard subset (near_dup vs hard_neg): **0.953** ✓ (> 0.65 threshold)
- Delta AUC (inv_ECS − Jaccard): **−0.046** ✗ (Jaccard is a perfect ceiling classifier; all hard_neg have Jaccard=0.0)
- **Verdict: PARTIAL** — ECS captures a strong real signal (MW p<0.001, Cohen's d=−2.19) but cannot complement perfect Jaccard on this synthetic dataset

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# loguru — NOT pre-installed on Colab
_pip('loguru==0.7.2')

# Core packages pre-installed on Colab; install locally to match Colab env
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'scipy==1.16.3', 'scikit-learn==1.6.1', 'matplotlib==3.10.0')

In [ ]:
import json
import sys
import math
from pathlib import Path

import numpy as np
from loguru import logger
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score
from sklearn.tree import DecisionTreeClassifier
import matplotlib.pyplot as plt

logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-f2202a-edit-clustering-score-spatial-edit-patte/main/round-2/evaluation-1/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception: pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f: return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
import os
data = load_data()
examples = data["datasets"][0]["examples"]
print(f"Loaded {len(examples)} examples from mini_demo_data.json")
print(f"Pair types: { {t: sum(1 for e in examples if e['metadata_pair_type']==t) for t in set(e['metadata_pair_type'] for e in examples)} }")

In [ ]:
# --- Config ---
# Bootstrap iterations. Original: B=2000. Set low for demo speed.
B = 200          # true original: 2000
RNG = np.random.default_rng(42)

## Helper Functions

Bootstrap utilities for AUC confidence intervals and delta-AUC.

In [ ]:
def bootstrap_auc(scores: np.ndarray, labels: np.ndarray, b: int = B) -> tuple:
    """Return (auc, ci_low, ci_high) via bootstrap."""
    n = len(labels)
    aucs = []
    for _ in range(b):
        idx = RNG.integers(0, n, size=n)
        sl, ll = scores[idx], labels[idx]
        if ll.sum() == 0 or ll.sum() == n:
            continue
        try:
            aucs.append(roc_auc_score(ll, sl))
        except Exception:
            continue
    aucs = np.array(aucs)
    return float(np.mean(aucs)), float(np.percentile(aucs, 2.5)), float(np.percentile(aucs, 97.5))


def bootstrap_delta_auc(scores1: np.ndarray, scores2: np.ndarray, labels: np.ndarray, b: int = B) -> tuple:
    """Bootstrap CI for delta AUC = auc(scores1) - auc(scores2)."""
    n = len(labels)
    deltas = []
    for _ in range(b):
        idx = RNG.integers(0, n, size=n)
        sl1, sl2, ll = scores1[idx], scores2[idx], labels[idx]
        if ll.sum() == 0 or ll.sum() == n:
            continue
        try:
            d = roc_auc_score(ll, sl1) - roc_auc_score(ll, sl2)
            deltas.append(d)
        except Exception:
            continue
    deltas = np.array(deltas)
    return float(np.mean(deltas)), float(np.percentile(deltas, 2.5)), float(np.percentile(deltas, 97.5))


def cv_auc(X: np.ndarray, y: np.ndarray) -> float:
    """5-fold stratified CV AUC."""
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    aucs = []
    for tr, te in skf.split(X, y):
        clf = LogisticRegression(max_iter=1000)
        clf.fit(X[tr], y[tr])
        prob = clf.predict_proba(X[te])[:, 1]
        if y[te].sum() == 0 or y[te].sum() == len(y[te]):
            continue
        aucs.append(roc_auc_score(y[te], prob))
    return float(np.mean(aucs))

## Data Extraction

Extract pair types, Jaccard, and ECS scores from examples. Build masks for near_dup, hard_neg, and the hard subset (near_dup vs hard_neg).

In [ ]:
# Extract fields
pair_types = [e["metadata_pair_type"] for e in examples]
jaccards = np.array([float(e["metadata_jaccard"]) for e in examples])
ecss = np.array([float(e["metadata_ecs"]) for e in examples])

# Inverted ECS score: lower ECS = more near-dup, so use -ECS as score
inv_ecs = -ecss

# --- Indices ---
nd_mask = np.array([p == "near_dup" for p in pair_types])
hn_mask = np.array([p == "hard_neg" for p in pair_types])
hard_mask = nd_mask | hn_mask
hard_labels = nd_mask[hard_mask].astype(int)  # 1=near_dup, 0=hard_neg

nd_idx = np.where(nd_mask)[0]
hn_idx = np.where(hn_mask)[0]

logger.info(f"near_dup={nd_mask.sum()} hard_neg={hn_mask.sum()} random={(~hard_mask).sum()}")
logger.info(f"Hard subset size: {hard_mask.sum()} (nd={nd_mask.sum()}, hn={hn_mask.sum()})")

## Metric 1: Inverted-ECS AUC on Hard Subset

The hard subset is near_dup vs hard_neg — the most challenging discrimination task. We compute AUC for both inv-ECS and Jaccard with bootstrap confidence intervals.

In [ ]:
# ========== METRIC 1: Inverted ECS AUC on hard subset ==========
hard_inv_ecs = inv_ecs[hard_mask]
hard_jaccard = jaccards[hard_mask]

ecs_auc, ecs_ci_lo, ecs_ci_hi = bootstrap_auc(hard_inv_ecs, hard_labels)
jac_auc_hard, jac_ci_lo, jac_ci_hi = bootstrap_auc(hard_jaccard, hard_labels)
logger.info(f"[M1] Inverted-ECS AUC on hard subset: {ecs_auc:.4f} CI [{ecs_ci_lo:.4f},{ecs_ci_hi:.4f}]")
logger.info(f"[M1] Jaccard AUC on hard subset: {jac_auc_hard:.4f} CI [{jac_ci_lo:.4f},{jac_ci_hi:.4f}]")

## Metric 2: Delta AUC — Does ECS Complement Jaccard?

We test whether adding ECS improves over Jaccard alone. Delta AUC = AUC(inv_ECS) − AUC(Jaccard) on the hard subset, with bootstrap CI.

In [ ]:
# ========== METRIC 2: Delta AUC (jaccard5+inv_ECS) vs jaccard5 alone ==========
# (a) Full dataset
full_labels_3 = np.array([1 if p == "near_dup" else 0 for p in pair_types])
# Use predict scores from method_out (trained scores)
pred_jaccard = np.array([float(e["predict_jaccard"]) for e in examples])
pred_ecs = np.array([float(e["predict_ecs"]) for e in examples])
pred_combined = np.array([float(e["predict_combined"]) for e in examples])

delta_full, d_full_lo, d_full_hi = bootstrap_delta_auc(pred_combined, pred_jaccard, full_labels_3)
logger.info(f"[M2a] Delta AUC (combined vs jaccard) full: {delta_full:.4f} CI [{d_full_lo:.4f},{d_full_hi:.4f}]")

# (b) Hard subset - use CV on hard subset with [jaccard, -ecs] vs jaccard alone
X_hard = np.column_stack([hard_jaccard, hard_inv_ecs])
X_jac_hard = hard_jaccard.reshape(-1, 1)

if hard_labels.sum() > 0 and hard_labels.sum() < len(hard_labels) and len(hard_labels) >= 5:
    auc_combined_hard = cv_auc(X_hard, hard_labels)
    auc_jac_hard_cv = cv_auc(X_jac_hard, hard_labels)
else:
    # Too few samples for 5-fold CV, use direct AUC
    clf_c = LogisticRegression(max_iter=1000).fit(X_hard, hard_labels)
    auc_combined_hard = roc_auc_score(hard_labels, clf_c.predict_proba(X_hard)[:, 1])
    clf_j = LogisticRegression(max_iter=1000).fit(X_jac_hard, hard_labels)
    auc_jac_hard_cv = roc_auc_score(hard_labels, clf_j.predict_proba(X_jac_hard)[:, 1])
    logger.warning("Hard subset too small for 5-fold CV; using in-sample AUC")

# Bootstrap delta for hard subset using raw scores
delta_hard, d_hard_lo, d_hard_hi = bootstrap_delta_auc(hard_inv_ecs, hard_jaccard, hard_labels)
logger.info(f"[M2b] Delta AUC (inv_ECS vs jaccard) hard: {delta_hard:.4f} CI [{d_hard_lo:.4f},{d_hard_hi:.4f}]")
logger.info(f"[M2b] Combined CV AUC on hard: {auc_combined_hard:.4f}, Jaccard CV AUC: {auc_jac_hard_cv:.4f}")

## Metrics 3–5: Source, Length Stratification, Decision Tree

Identify the dataset source, stratify by ECS-proxied length tercile, and fit a depth-2 decision tree to see which feature the model relies on.

In [ ]:
# ========== METRIC 3: Dataset source labeling ==========
source = "synthetic_vocab_template"
logger.info(f"[M3] Dataset source: {source} (NOT Wikipedia-derived)")

# ========== METRIC 4: Length-stratified AUC ==========
# Estimate length from input string: "pair_type=X jaccard=Y ecs=Z"
# We don't have actual word counts, so we derive proxy from ECS magnitude
# ECS = var/mean of inter-edit-gap positions → larger document → more absolute gap values
# Instead, use ECS as proxy (monotone with length) — bin hard subset by ECS range
# Better: use rank-based tercile on hard subset ECS values
hard_ecs_vals = ecss[hard_mask]
tercile_bounds = np.percentile(hard_ecs_vals, [33.3, 66.7])
tercile_labels_arr = np.digitize(hard_ecs_vals, tercile_bounds)  # 0, 1, 2

length_strat = {}
for t, name in enumerate(["short", "medium", "long"]):
    tmask = tercile_labels_arr == t
    if tmask.sum() < 2 or hard_labels[tmask].sum() == 0 or hard_labels[tmask].sum() == tmask.sum():
        length_strat[name] = {"n": int(tmask.sum()), "note": "insufficient_class_diversity"}
        continue
    ecs_t_auc, _, _ = bootstrap_auc(hard_inv_ecs[tmask], hard_labels[tmask])
    jac_t_auc, _, _ = bootstrap_auc(hard_jaccard[tmask], hard_labels[tmask])
    length_strat[name] = {
        "n": int(tmask.sum()),
        "inv_ecs_auc": float(ecs_t_auc),
        "jaccard_auc": float(jac_t_auc),
    }
logger.info(f"[M4] Length-stratified AUC: {length_strat}")

# ========== METRIC 5: Depth-2 Decision Tree ==========
dt_result = {}
if len(hard_labels) >= 4 and hard_labels.sum() > 0:
    X_tree = np.column_stack([hard_jaccard, hard_inv_ecs])
    dt = DecisionTreeClassifier(max_depth=2, random_state=42)
    dt.fit(X_tree, hard_labels)
    tree = dt.tree_
    dt_result = {
        "feature_names": ["jaccard", "inv_ecs"],
        "n_nodes": int(tree.node_count),
        "root_feature": int(tree.feature[0]),
        "root_threshold": float(tree.threshold[0]),
        "root_impurity": float(tree.impurity[0]),
        "feature_importances": {
            "jaccard": float(dt.feature_importances_[0]),
            "inv_ecs": float(dt.feature_importances_[1]),
        },
    }
    # Extract child splits if they exist
    left_child = tree.children_left[0]
    right_child = tree.children_right[0]
    if left_child != -1:
        dt_result["left_split"] = {
            "feature": int(tree.feature[left_child]),
            "threshold": float(tree.threshold[left_child]),
        }
    if right_child != -1:
        dt_result["right_split"] = {
            "feature": int(tree.feature[right_child]),
            "threshold": float(tree.threshold[right_child]),
        }
logger.info(f"[M5] Decision tree: {dt_result}")

## Metric 6: Mann-Whitney U Test

Non-parametric test for whether near_dup pairs have significantly lower ECS than hard_neg pairs. Also computes Cohen's d on log-transformed ECS (effect size).

In [ ]:
# ========== METRIC 6: Mann-Whitney U statistics ==========
nd_ecs = ecss[nd_mask]
hn_ecs = ecss[hn_mask]

mw_stat, mw_p = stats.mannwhitneyu(nd_ecs, hn_ecs, alternative="two-sided")
# MW with correct direction: nd should have LOWER ecs
mw_stat_less, mw_p_less = stats.mannwhitneyu(nd_ecs, hn_ecs, alternative="less")

median_nd = float(np.median(nd_ecs))
median_hn = float(np.median(hn_ecs))
median_ratio = float(median_nd / median_hn) if median_hn > 0 else float("nan")

# Cohen's d on log-IoD
log_nd = np.log(nd_ecs + 1)
log_hn = np.log(hn_ecs + 1)
pooled_std = math.sqrt((log_nd.var() + log_hn.var()) / 2)
cohens_d = float((log_nd.mean() - log_hn.mean()) / pooled_std) if pooled_std > 0 else 0.0

logger.info(f"[M6] MW U={mw_stat:.1f} p={mw_p:.6f} (two-sided), p_less={mw_p_less:.6f}")
logger.info(f"[M6] Median IoD near_dup={median_nd:.3f}, hard_neg={median_hn:.3f}, ratio={median_ratio:.3f}")
logger.info(f"[M6] Cohen's d on log-IoD={cohens_d:.3f}")

## Boilerplate Augmentation & Verdict

Simulate a regime where hard_neg pairs have boilerplate overlap (Jaccard += 0.35). Tests whether ECS adds value when Jaccard is no longer a ceiling classifier.

In [ ]:
# ========== METRIC 2 supplemental: Boilerplate augmented hard subset ==========
# Simulate: hard_neg jaccard += 0.35 (boilerplate overlap), near_dup unchanged
jac_bpl = hard_jaccard.copy()
jac_bpl[hard_labels == 0] = np.minimum(1.0, jac_bpl[hard_labels == 0] + 0.35)
# ECS unchanged (boilerplate doesn't cluster edits)
bpl_inv_ecs = hard_inv_ecs.copy()

bpl_ecs_auc, bpl_ecs_ci_lo, bpl_ecs_ci_hi = bootstrap_auc(bpl_inv_ecs, hard_labels)
bpl_jac_auc, bpl_jac_ci_lo, bpl_jac_ci_hi = bootstrap_auc(jac_bpl, hard_labels)
bpl_delta, bpl_d_lo, bpl_d_hi = bootstrap_delta_auc(bpl_inv_ecs, jac_bpl, hard_labels)
logger.info(f"[M2-BPL] Boilerplate-augmented: ECS AUC={bpl_ecs_auc:.4f}, Jaccard AUC={bpl_jac_auc:.4f}, delta={bpl_delta:.4f}")

# ========== METRIC 7: Verdict ==========
ecs_auc_ok = ecs_auc > 0.65
delta_hard_ok = delta_hard > 0.03
jaccard_ceiling = jac_auc_hard >= 0.999

if ecs_auc_ok and delta_hard_ok:
    verdict = "CONFIRMED"
elif ecs_auc_ok or delta_hard_ok:
    verdict = "PARTIAL"
else:
    verdict = "DISCONFIRMED"

logger.info(f"[M7] Verdict: {verdict} (ecs_auc_ok={ecs_auc_ok}, delta_hard_ok={delta_hard_ok}, jaccard_ceiling={jaccard_ceiling})")

print(f"\n{'='*60}")
print(f"VERDICT: {verdict}")
print(f"  Inverted-ECS AUC on hard subset: {ecs_auc:.4f} (>{0.65}? {ecs_auc_ok})")
print(f"  Delta AUC (inv_ECS-jaccard) hard: {delta_hard:.4f} (>0.03? {delta_hard_ok})")
print(f"  Jaccard ceiling: {jaccard_ceiling}")
print(f"  MW p (nd<hn ECS): {mw_p_less:.4f}")
print(f"  Hard_neg n={int(hn_mask.sum())} (NOTE: very small, statistical power limited)")
print(f"{'='*60}")

## Visualization: Key Results Summary

Plot the ECS distributions by pair type and the AUC comparison.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# --- Plot 1: ECS distribution by pair type ---
ax = axes[0]
nd_ecs_vals = ecss[nd_mask]
hn_ecs_vals = ecss[hn_mask]
rand_ecs_vals = ecss[~hard_mask]
ax.hist(nd_ecs_vals, bins=20, alpha=0.6, label=f"near_dup (n={nd_mask.sum()})", color="steelblue")
ax.hist(rand_ecs_vals, bins=20, alpha=0.5, label=f"random (n={(~hard_mask).sum()})", color="gray")
for v in hn_ecs_vals:
    ax.axvline(v, color="red", alpha=0.8, linewidth=1.5)
ax.axvline(hn_ecs_vals[0], color="red", alpha=0.8, linewidth=1.5, label=f"hard_neg (n={hn_mask.sum()})")
ax.set_xlabel("ECS (Inter-of-Dispersion of edit gaps)")
ax.set_ylabel("Count")
ax.set_title("ECS Distribution by Pair Type")
ax.legend(fontsize=8)

# --- Plot 2: AUC comparison bar chart ---
ax = axes[1]
metrics = ["inv-ECS\n(hard)", "Jaccard\n(hard)", "inv-ECS\n(boilerplate)", "Jaccard\n(boilerplate)"]
aucs_vals = [ecs_auc, jac_auc_hard, bpl_ecs_auc, bpl_jac_auc]
colors = ["steelblue", "orange", "steelblue", "orange"]
bars = ax.bar(metrics, aucs_vals, color=colors, alpha=0.8)
ax.axhline(0.65, color="green", linestyle="--", linewidth=1.5, label="ECS threshold (0.65)")
ax.axhline(1.0, color="red", linestyle=":", linewidth=1, label="Ceiling (1.0)")
ax.set_ylim(0, 1.1)
ax.set_ylabel("AUC")
ax.set_title("AUC Comparison")
ax.legend(fontsize=8)
for bar, val in zip(bars, aucs_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, f"{val:.3f}", ha="center", fontsize=9)

# --- Plot 3: Summary metrics table ---
ax = axes[2]
ax.axis("off")
summary_data = [
    ["Metric", "Value", "Pass?"],
    ["inv-ECS AUC (hard)", f"{ecs_auc:.4f}", "✓" if ecs_auc_ok else "✗"],
    ["Jaccard AUC (hard)", f"{jac_auc_hard:.4f}", "ceiling"],
    ["Delta AUC (hard)", f"{delta_hard:.4f}", "✓" if delta_hard_ok else "✗"],
    ["MW p (nd<hn)", f"{mw_p_less:.5f}", "✓"],
    ["Cohen's d (log-ECS)", f"{cohens_d:.3f}", "large"],
    ["Verdict", verdict, ""],
]
tbl = ax.table(cellText=summary_data[1:], colLabels=summary_data[0], loc="center", cellLoc="center")
tbl.auto_set_font_size(False)
tbl.set_fontsize(10)
tbl.scale(1.2, 1.6)
ax.set_title("Key Results Summary", pad=20)

plt.tight_layout()
plt.savefig("ecs_vs_jaccard_results.png", dpi=100, bbox_inches="tight")
plt.show()
print("Figure saved to ecs_vs_jaccard_results.png")